# Environment setup

This notebook sets up everything you need for the hackathon:

1. Create a Python 3.11 **virtual environment** for the project
2. Install the required libraries from `requirements.txt`
3. Register the environment as a **Jupyter kernel**
4. Check that the **GPU** is visible from PyTorch
5. **Download** the dataset

Run the cells from top to bottom with `Shift + Enter`.

<!-- TODO(organisers): add any cluster-specific notes here (modules to load, shared data paths, quotas, ...). -->

## Part 1: Create the environment

> Run Part 1 with the **default kernel** that JupyterLab gives you. You'll switch to the new kernel in Part 2.

### What is a virtual environment?

<!-- TODO(organisers): short explanation of why we isolate dependencies per project. -->

First, let's see which Python we're running and where the repository is.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve()  # this notebook lives in the repository root
VENV = REPO / ".venv"
PY = VENV / "bin" / "python"

print("Python executable:", sys.executable)
print("Python version:   ", sys.version.split()[0])
print("Repository root:  ", REPO)
assert (REPO / "requirements.txt").exists(), "Open this notebook from the root folder of the repository."

### 1.1 Create the environment

The default Python on this server is too new for TensorFlow 2.15, so we use [`uv`](https://docs.astral.sh/uv/) to download **Python 3.11** and create the environment with it.

We create it in `.venv/` inside the repository. That folder is in `.gitignore`, so it never gets committed.

The `!` at the start of a line tells Jupyter to run it as a **shell command**. `{VENV}` inserts the value of the Python variable.

In [ ]:
!{sys.executable} -m pip install --quiet --user uv
!rm -rf {VENV}
!{sys.executable} -m uv venv --python 3.11 --seed {VENV}
!{PY} --version

### 1.2 Install the dependencies

This takes a few minutes. TensorFlow is the largest download.

In [ ]:
!{PY} -m pip install --upgrade pip
!{PY} -m pip install -r {REPO}/requirements.txt

### 1.3 Register the environment as a Jupyter kernel

Jupyter can only use your new environment once it's registered as a *kernel*.

In [ ]:
!{PY} -m ipykernel install --user --name adatl1 --display-name "Python (adatl1)"
!jupyter kernelspec list

### 1.4 Switch kernel

Now switch this notebook over to the new environment:

**Kernel → Change Kernel… → `Python (adatl1)`**

(You may need to refresh the browser tab before it appears.) Then carry on with Part 2 below. **Don't re-run Part 1.**

<details>
<summary>Prefer the terminal? Click here for the equivalent commands.</summary>

```bash
cd hackathon_ADatL1
python -m pip install --user uv
python -m uv venv --python 3.11 --seed .venv
.venv/bin/python -m pip install --upgrade pip
.venv/bin/python -m pip install -r requirements.txt
.venv/bin/python -m ipykernel install --user --name adatl1 --display-name "Python (adatl1)"
```
</details>

## Part 2: Verify the environment

> From here on, the kernel name in the top-right corner should say **Python (adatl1)**.

In [ ]:
import sys
print(sys.executable)
assert ".venv" in sys.executable, "You are not on the adatl1 kernel yet: Kernel -> Change Kernel -> Python (adatl1)"

In [ ]:
import numpy, pandas, pyarrow, awkward, sklearn, matplotlib, torch, yaml

for mod in (numpy, pandas, pyarrow, awkward, sklearn, matplotlib, torch, yaml):
    print(f"{mod.__name__:12s} {mod.__version__}")

### 2.1 GPU check

<!-- TODO(organisers): describe the GPU resources available (type, how many per team, sharing etiquette). -->

In [ ]:
!nvidia-smi

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  [{i}] {torch.cuda.get_device_name(i)}")
    device = torch.device("cuda")
else:
    print("No GPU found; falling back to CPU. Ask an organiser if you expected one!")
    device = torch.device("cpu")

# Quick smoke test: a matrix multiplication on the device
a = torch.randn(4096, 4096, device=device)
print("matmul OK:", (a @ a).shape, "on", device)

## Part 3: Download the data

The dataset is hosted on the [Hugging Face Hub](https://huggingface.co/datasets/CERN/anomaly_detection_cmsl1t).
The full record is tens of GB, so we only download what the config files in `config/` need.

| Config | Downloads | Approx. size |
|---|---|---|
| `config/baseline.yml` | one zero-bias run + 4 signals | ~1.1 GB |
| `config/combined.yml` | both zero-bias runs + 4 signals | ~2 GB |
| `config/simulation.yml` | simulated zero bias + 4 signals | ~0.4 GB |

Files go to `data/` in the repository. If the organisers have put a shared copy somewhere, change
`root:` in the config files to that path instead and skip the download.

<!-- TODO(organisers): mention any shared data path. -->

In [ ]:
from dataloader import download_config

download_config("config/baseline.yml")

In [ ]:
# Optional: the other experiments
# download_config("config/combined.yml")
# download_config("config/simulation.yml")

In [ ]:
# Sanity check: load a few events
import dataset

events = dataset.load_events("ZB_run396102", "valid", max_events=1000)
print(len(events), "events loaded; fields:", events.fields)

🎉 **You're all set!** Continue with [`explore_data.ipynb`](explore_data.ipynb).